# 03 Self-Attention 自注意力机制

前面两节已经铺好了两块基础。

第一块：Attention 的基本流程。

```text
相关性分数 -> Softmax -> 注意力权重 -> 加权汇总
```

第二块：$Q$、$K$、$V$ 的分工。

- $Q$：当前我想找什么
- $K$：每份信息拿什么来匹配
- $V$：真正被加权汇总的内容

这一节学习 Self-Attention，也叫自注意力机制。

这一节先解决一个问题：

```text
一句话内部的每个 token，如何参考同一句话里的其他 token，更新自己的表示？
```

## 1. Self-Attention 的 self 是什么意思

Self-Attention 里的 self，可以先理解成“自己这一组输入内部”。

也就是说：

- $Q$ 来自这句话本身。
- $K$ 来自这句话本身。
- $V$ 也来自这句话本身。

模型不是拿一句话去看另一句话，而是让一句话内部的各个位置互相参考。

例如：

```text
小明 把 复习资料 交给 小红 因为 她 明天 要 考试
```

当模型处理“她”这个位置时，它可以参考同一句话里的“小明”“小红”“考试”等位置。

这就是 self 的含义：

```text
在同一份输入内部计算注意力。
```

## 2. 为什么需要 Self-Attention

一个 token 的原始向量，通常只代表它自己当前的表示。

但是在句子里，词的含义经常依赖上下文。

例如“她”这个词，单独看没有足够信息。

只有结合上下文，才能判断它更可能指“小红”。

所以模型需要让每个位置的表示变得更有上下文信息。

Self-Attention 要做的就是：

```text
让每个位置都根据整句话的信息，更新自己的表示。
```

更新之后，“她”的新表示就不再只是“她”这个字本身，而是融合了它和上下文之间的关系。

## 3. 先看一个长度为 4 的小序列

为了不被长句子绕晕，我们先看 4 个 token：

- $x_{1}$：小明
- $x_{2}$：资料
- $x_{3}$：小红
- $x_{4}$：她

每个 token 先有一个输入向量：

$$
\mathbf{x}_1,\mathbf{x}_2,\mathbf{x}_3,\mathbf{x}_4
$$

Self-Attention 会让每个位置都重新生成一个新表示：

$$
\mathbf{c}_1,\mathbf{c}_2,\mathbf{c}_3,\mathbf{c}_4
$$

其中 $\mathbf{c}_4$ 就是“她”这个位置融合上下文后的新表示。

这一节的关键是理解：

每个 $c_{i}$ 都不是只来自 $x_{i}$ 自己，而是来自所有位置的加权汇总。

## 4. 第一步：每个位置先生成自己的 $Q$、$K$、$V$

对每个输入位置 $\mathbf{x}_i$，模型会生成三种表示：

$$
\mathbf{q}_i=\mathbf{x}_i\mathbf{W}_Q
$$

$$
\mathbf{k}_i=\mathbf{x}_i\mathbf{W}_K
$$

$$
\mathbf{v}_i=\mathbf{x}_i\mathbf{W}_V
$$

如果有 4 个 token，就会得到：

$$
\begin{aligned}
x_{1} \rightarrow q_{1}, k_{1}, v_{1} \\
x_{2} \rightarrow q_{2}, k_{2}, v_{2} \\
x_{3} \rightarrow q_{3}, k_{3}, v_{3} \\
x_{4} \rightarrow q_{4}, k_{4}, v_{4}
\end{aligned}
$$

这一步还没有开始互相看。

它只是给每个位置准备好三种身份：

- $q_{i}$：我想找什么
- $k_{i}$：别人怎么匹配我
- $v_{i}$：我能提供什么内容

## 5. 第二步：一个位置去看所有位置

现在只看第 4 个位置，也就是“她”。

第 4 个位置拿自己的 Query：

$$
\mathbf{q}_4
$$

去和所有位置的 Key 做匹配：

$$
\mathbf{k}_1,\mathbf{k}_2,\mathbf{k}_3,\mathbf{k}_4
$$

得到 4 个相关性分数：

$$
s_{4,1}=\mathbf{q}_4\cdot\mathbf{k}_1
$$

$$
s_{4,2}=\mathbf{q}_4\cdot\mathbf{k}_2
$$

$$
s_{4,3}=\mathbf{q}_4\cdot\mathbf{k}_3
$$

$$
s_{4,4}=\mathbf{q}_4\cdot\mathbf{k}_4
$$

读法是：

- $s_{4,1}$：第 4 个位置看第 1 个位置的分数
- $s_{4,2}$：第 4 个位置看第 2 个位置的分数
- $s_{4,3}$：第 4 个位置看第 3 个位置的分数
- $s_{4,4}$：第 4 个位置看第 4 个位置自己的分数

这个地方就是 Self-Attention 的核心动作：一个位置直接和所有位置计算关系。

## 6. 第三步：分数变成权重

假设第 4 个位置看所有位置的分数是：

$$
\begin{aligned}
\mathrm{\mathrm{score}}_4 &= [0.3, 0.2, 2.1, 0.8]
\end{aligned}
$$

这些只是原始分数，还不是权重。

经过 Softmax 后，可能得到：

$$
\begin{aligned}
\alpha_{4} &= [0.10, 0.09, 0.60, 0.21]
\end{aligned}
$$

这表示第 4 个位置更新自己时：

```text
参考第 1 个位置的比例：0.10
参考第 2 个位置的比例：0.09
参考第 3 个位置的比例：0.60
参考第 4 个位置自己的比例：0.21
```

如果第 3 个位置是“小红”，那么这组权重就表示：“她”这个位置更重点参考“小红”。

注意，这些数字只是帮助理解，不是人为规定的。真实权重由模型根据输入动态算出来。

## 7. 第四步：用权重汇总 Value

得到权重之后，第 4 个位置要更新自己的表示。

它不是去加权 Key，而是加权 Value：

$$
\mathbf{c}_4=0.10\mathbf{v}_1+0.09\mathbf{v}_2+0.60\mathbf{v}_3+0.21\mathbf{v}_4
$$

$\mathbf{c}_4$ 就是第 4 个位置的新表示。

如果 $\mathbf{v}_3$ 对应“小红”的内容，而且权重最大，那么 $\mathbf{c}_4$ 就会明显融合“小红”的信息。

这就是为什么 Self-Attention 能让一个词的表示带上上下文。

一句话总结：

当前位置用自己的 $Q$ 去匹配所有 $K$，得到权重，再用权重汇总所有 $V$，生成当前位置的新表示。

## 8. 每个位置都会做同样的事

刚才只看了第 4 个位置。

但 Self-Attention 里，每个位置都会这样做。

第 1 个位置会用 $\mathbf{q}_1$ 看所有 Key，得到 $\mathbf{c}_1$。

第 2 个位置会用 $\mathbf{q}_2$ 看所有 Key，得到 $\mathbf{c}_2$。

第 3 个位置会用 $\mathbf{q}_3$ 看所有 Key，得到 $\mathbf{c}_3$。

第 4 个位置会用 $\mathbf{q}_4$ 看所有 Key，得到 $\mathbf{c}_4$。

写成一条线就是：

$$
\begin{aligned}
x_{1} \rightarrow c_{1} \\
x_{2} \rightarrow c_{2} \\
x_{3} \rightarrow c_{3} \\
x_{4} \rightarrow c_{4}
\end{aligned}
$$

注意：这里不是每个位置只处理自己。

每个 $c_i$ 都是第 $i$ 个位置参考整句话之后得到的新表示。

## 9. 为什么会得到 $N \times N$ 的注意力表

如果输入有 $N$ 个位置，每个位置都要看所有 $N$ 个位置。

那么分数就会形成一张表：

$$
\begin{array}{c|ccccc}
 & k_1 & k_2 & k_3 & \cdots & k_N \\
q_1 & s_{11} & s_{12} & s_{13} & \cdots & s_{1N} \\
q_2 & s_{21} & s_{22} & s_{23} & \cdots & s_{2N} \\
q_3 & s_{31} & s_{32} & s_{33} & \cdots & s_{3N} \\
\vdots & \vdots & \vdots & \vdots & \ddots & \vdots \\
q_N & s_{N1} & s_{N2} & s_{N3} & \cdots & s_{NN}
\end{array}
$$

这张表的形状就是：

$$
N\times N
$$

每一行表示一个 Query 在看所有 Key。

每一行经过 Softmax，就变成这个位置的注意力权重。

所以 $N\times N$ 不是凭空来的。

它来自：

$N$ 个位置，每个位置都看 $N$ 个位置。

## 10. Self-Attention 和普通 Attention 的区别

普通 Attention 更宽泛。

它只要求：

用 $Q$ 去匹配 $K$，再按权重汇总 $V$。

$Q$、$K$、$V$ 不一定来自同一份输入。

Self-Attention 更具体。

它要求：

$Q$、$K$、$V$ 都来自同一份输入。

所以可以这样对比：

```text
Attention：一种通用的 QKV 加权汇总机制。
Self-Attention：同一份输入内部自己看自己。
```

后面如果学到 Cross-Attention，会看到另一种情况：$Q$ 来自一份输入，$K$ 和 $V$ 来自另一份输入。

但这一节先只关注 Self-Attention。

## 11. Self-Attention 和 CNN 的区别

CNN 处理图像时，通常先看局部窗口。

比如一个 $3 \times 3$ 卷积核，一次只看周围一小块区域。

Self-Attention 则让每个位置可以直接看所有位置。

可以先这样理解：

```text
CNN：先看附近，再通过多层逐渐扩大范围。
Self-Attention：一开始就允许每个位置直接和所有位置计算关系。
```

这也是为什么 Self-Attention 很适合处理需要长距离关系的序列。

但它也有代价。

因为每个位置都看所有位置，所以关系数量是 $N\times N$。

序列越长，计算量就越大。

## 12. Self-Attention 输出的形状是什么

假设输入是一句话，有 $N$ 个位置，每个位置原来用 $D$ 维表示。

输入可以写成：

$$
N\times D
$$

Self-Attention 会为每个位置生成一个新表示。

所以输出仍然有 $N$ 个位置：

- 输入：$x_{1}$, $x_{2}$, ..., $x_{N}$
- 输出：$c_{1}$, $c_{2}$, ..., $c_{N}$

如果输出维度也是 $D$，那输出形状仍然可以是：

$$
N\times D
$$

如果加上 batch，一批输入就是：

$$
B\times N\times D
$$

很多时候 Self-Attention 层的输入和输出形状保持一致。

但每个位置的内容已经变了：

```text
原来：每个位置主要是自己的表示。
现在：每个位置融合了上下文信息。
```

## 13. 为什么还需要位置编码

Self-Attention 有一个特点：它主要根据内容之间的相关性来计算权重。

如果只看 Attention 计算本身，它不天然知道谁在前、谁在后。

例如下面两句话：

```text
我 喜欢 你
你 喜欢 我
```

词集合很像，但顺序不同，意思不同。

所以 Transformer 里通常需要加入位置信息，也就是后面会学的位置编码。

这一节先不用展开位置编码，只要先知道：

```text
Self-Attention 擅长建模位置之间的关系。
但模型还需要额外知道位置信息。
```

## 14. 这一节先不要急着掌握什么

这一节是 Self-Attention 的概念课。

现在先不要急着掌握：

```text
PyTorch 代码怎么写
Multi-Head Attention 怎么拆多个头
完整 Transformer Encoder 怎么搭建
mask 在不同任务里怎么用
位置编码的具体公式
```

这些后面会分开讲。

现在只需要把 Self-Attention 的核心画面立住：

```text
同一句话里，每个位置都用自己的 Query 去看所有位置的 Key，得到权重，再按权重汇总所有位置的 Value。
```

## 15. 本节小结

这一节先记住：

1. Self-Attention 表示同一份输入内部自己计算注意力。
2. 每个位置都会从自己的输入表示生成 $Q$、$K$、$V$。
3. 一个位置用自己的 Query 去匹配所有位置的 Key。
4. 分数经过 Softmax 变成注意力权重。
5. 注意力权重作用在所有 Value 上，得到这个位置的新表示。
6. 每个位置都会做同样的计算，所以会得到 $N\times N$ 的注意力表。
7. Self-Attention 的输出仍然保留每个位置，只是每个位置已经融合了上下文。
8. Self-Attention 本身还需要配合位置信息，后面会学习位置编码。

## 16. 自测问题

1. Self-Attention 里的 self 指什么？
2. 为什么一个 token 的表示需要融合上下文？
3. 对一个输入位置 $\mathbf{x}_i$，模型会生成哪三个表示？
4. 第 4 个位置用 $\mathbf{q}_4$ 去匹配哪些 Key？
5. $s_{4,3}$ 可以怎么读？
6. 为什么原始分数还要经过 Softmax？
7. 为什么最后汇总的是 Value，而不是 Key？
8. 为什么 Self-Attention 会得到 $N\times N$ 的分数表？
9. Self-Attention 和普通 Attention 有什么区别？
10. Self-Attention 和 CNN 的关注方式有什么不同？
11. Self-Attention 输出为什么通常还保留 $N$ 个位置？
12. 为什么后面还要学习位置编码？